In [ ]:
from langchain_community.document_loaders import YoutubeLoader
from langchain_core.documents import Document 
import yt_dlp
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api.proxies import WebshareProxyConfig
import concurrent.futures
import re
import json
import jsonlines

/home/nando/anaconda3/envs/ds_cont/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from urls import list_urls
url_videos = list_urls

In [16]:
import requests
import yt_dlp

transcriptions = {}

def video_processing(video_url):
    ydl_opts = {
        'quiet': True,
        'skip_download': True,
        'writeautomaticsub': True,
        'subtitleslangs': ['es'],
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info_dict = ydl.extract_info(video_url, download=False)
        es_subs = info_dict.get('automatic_captions', {}).get('es', [])

    for fmt in es_subs:
        if fmt.get('ext') == 'json3':
            data = requests.get(fmt['url']).json()
            words = []
            for event in data.get('events', []):
                for seg in event.get('segs', []):
                    text = seg.get('utf8', '').strip()
                    if text:
                        words.append(text)
            transcriptions[video_url] = " ".join(words)
            return

video_processing("j_oHko0S_Cc")
print(transcriptions["j_oHko0S_Cc"])

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
asd = '''
v97 10 2 Gestión de Importaciones \n En el presente video aprenderás a gestionar las importaciones realizadas. \n  Gestión de importaciones. De ese CONT Cloud, cuenta con múltiples opciones para el registro de información contable de operaciones de compras ventas caja entre otras. Dentro de estas opciones encontramos la función importar desde Excel, importar desde XML, importar desde Siri, disponibles tanto en el módulo de ventas como en el módulo de compras. Sin embargo, puede suceder que luego de importar esta información, decemos eliminar la de forma masiva, sin tener que eliminar una por una laso operaciones desde los listados. Para esto, el sistema nos ofrece una herramienta muy útil, el módulo de gestión de importaciones, para acceder a este módulo nos dirigimos a la sección utilitarios gestionar importaciones. Aquí se mostrarán todas las importaciones que hayamos realizado. Ya sean del módulo de compras, de ventas o de otros módulos que permitan importar datos. El Estado incluye datos como el código de importación, descripción, el número de operaciones procesadas e importadas, la observación colocada al momento del registro y el usuario que realizó la importación. Si deseamos eliminar los registros importados a la vez, podemos hacerlo de forma masiva desde esta misma ventana, sin necesidad de borrar cada registro individualmente. Por ejemplo, supongamos que queremos eliminar las ventas importadas en el periodo de enero. Ubicamos la importación correspondiente y presionamos el botón eliminar. El sistema nos mostrará una alerta de confirmación debido a que esta operación no se puede revertir. Confirmamos presionando nuevamente eliminar y el sistema procederá a borrar todas las operaciones que fueron registradas en esa importación. De esta manera, logramos una gestión más eficiente y rápida de las importaciones realizadas dentro del sistema. Con este tema, concluyen nuestro curso de Deysé con Cloud nos despedimos hasta las próximas actualizaciones."
'''

In [ ]:
for elem in a:
    print(elem)

In [ ]:
import time

MAX_RETRIES = 5

documents = []

for video_url in url_videos:
    
    success = False
    
    for attempt in range(MAX_RETRIES):
        try:
            result = video_procesing(video_url)
            documents.append(result)
            
            success = True
            print(f"Successfully processed: {video_url}")
            break 
            
        except Exception as e:
            print(f"Attempt {attempt + 1} failed for {video_url}: {e}")
            
            time.sleep(1) 
    
    if not success:
        print(f"FAILED to process {video_url} after {MAX_RETRIES} attempts.")

print(f"Finished processing. Total documents collected: {len(documents)}")

In [ ]:
documents

In [ ]:
file_path = "Doc.jsonl"
print(f"Saving documents to {file_path}...")
with jsonlines.open(file_path, mode='w') as writer:
    for doc in documents:
        # Convert the Document object to a dictionary
        writer.write(doc.dict())
print("Save complete.")

In [ ]:
len(url_videos)

In [ ]:
documents

# Final test

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, NoTranscriptFound, TranscriptsDisabled
from urllib.parse import urlparse, parse_qs

def transcribe(video_url: str) -> str:
    video_id = parse_qs(urlparse(video_url).query)["v"][0]

    try:
        transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)  # classmethod ✅
        try:
            transcript = transcript_list.find_manually_created_transcript()
        except NoTranscriptFound:
            transcript = transcript_list.find_generated_transcript(["en"])

        segments = transcript.fetch()
        return " ".join(seg["text"] for seg in segments)  # dict access ✅

    except (NoTranscriptFound, TranscriptsDisabled):
        raise RuntimeError("No transcript available for this video.")

print(transcribe("https://www.youtube.com/watch?v=tQGFKa9wau4"))

tQGFKa9wau4


RuntimeError: No transcript available for this video.

In [ ]:
loaded_docs = []
file_path = "Doc.jsonl"
with jsonlines.open(file_path, mode='r') as reader:
    for doc_dict in reader:
        loaded_docs.append(Document(**doc_dict))

print(f"Loaded {len(loaded_docs)} documents.")
print(f"First document content: {loaded_docs[0].page_content}")
print(f"First document metadata: {loaded_docs[0].metadata}")

Loaded 96 documents.
First document content: v97 10 2 Gestión de Importaciones 
 En el presente video aprenderás a gestionar las importaciones realizadas. 
  Gestión de importaciones. De ese CONT Cloud, cuenta con múltiples opciones para el registro de información contable de operaciones de compras ventas caja entre otras. Dentro de estas opciones encontramos la función importar desde Excel, importar desde XML, importar desde Siri, disponibles tanto en el módulo de ventas como en el módulo de compras. Sin embargo, puede suceder que luego de importar esta información, decemos eliminar la de forma masiva, sin tener que eliminar una por una laso operaciones desde los listados. Para esto, el sistema nos ofrece una herramienta muy útil, el módulo de gestión de importaciones, para acceder a este módulo nos dirigimos a la sección utilitarios gestionar importaciones. Aquí se mostrarán todas las importaciones que hayamos realizado. Ya sean del módulo de compras, de ventas o de otros módulos que

In [14]:
for document in loaded_docs:
    current_url = document.metadata["source"]
    transcription = transcribe(current_url)
    document.page_content = transcription

In [15]:
file_path = "Doc_2.jsonl"
print(f"Saving documents to {file_path}...")
with jsonlines.open(file_path, mode='w') as writer:
    for doc in loaded_docs:
        # Convert the Document object to a dictionary
        writer.write(doc.dict())
print("Save complete.")

Saving documents to Doc_2.jsonl...
Save complete.


/tmp/ipykernel_1257283/3520260001.py:6: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  writer.write(doc.dict())
